# 🏔️ Generate and Load Sample Data - Snowflake Native

This notebook generates realistic sample data for the Strava ML demo and loads it directly into Snowflake tables.

## What this notebook does:
- **Creates 2,000 athletes** with realistic profiles and fitness levels
- **Generates 10,000 activities** with realistic metrics and fraud indicators  
- **Creates 10,000 challenge participations** across brand partners
- **Generates 10,000 subscription records** with churn risk scores
- **Loads all data directly** into Snowflake tables

## Prerequisites:
1. Run `01_env_setup.sql` first to create the environment and tables
2. Ensure you're connected to the `STRAVA_DEMO_SAMPLE.RAW_DATA` schema
3. Running in **Snowflake Notebooks** environment
4. **Packages installed**: Use package selector to add:
   - `pandas` (if not pre-installed)
   - `numpy` (if not pre-installed)

## Data Features:
- **Fraud Detection Ready**: Activities include fraudulent patterns for ML training
- **Realistic Distributions**: Data follows real-world Strava patterns
- **Time-based**: 50% of data in past 30 days for recency analysis


In [ ]:
# 🏔️ Setup for Snowflake Notebooks
print("🏔️ Running in Snowflake Notebooks environment")

# Import required libraries (pre-installed in Snowflake)
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Get Snowflake session (works in both Snowflake Notebooks and locally in Cursor)
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("✅ Running in Snowflake Notebooks")
except:
    from snowflake_connection import get_local_session
    session = get_local_session()
    print("✅ Running locally in Cursor")

print("✅ Connected to Snowflake")
print(f"🏢 Account: {session.get_current_account()}")
print(f"👤 User: {session.get_current_user()}")
print(f"🏗️ Warehouse: {session.get_current_warehouse()}")
print(f"🗃️ Database: {session.get_current_database()}")
print(f"📁 Schema: {session.get_current_schema()}")

# Ensure we're in the right context
session.use_database("STRAVA_DEMO_SAMPLE")
session.use_schema("RAW_DATA")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print(f"🎯 Ready to work with {session.get_current_database()}.{session.get_current_schema()}")
print("="*60)
import sys
import subprocess

def install_package(package):
    """Install a package using pip"""
    try:
        # Handle special import name mappings
        import_name = package.split('==')[0].replace('-', '_')
        if package.startswith('snowflake-snowpark-python'):
            import_name = 'snowflake.snowpark'
        
        __import__(import_name)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

# List of required packages for data generation
required_packages = [
    "snowflake-snowpark-python",
    "pandas",
    "numpy"
]

print("🔍 Checking and installing required packages...")
for package in required_packages:
    install_package(package)

print("✅ All packages ready!")
print("="*60)

# Import required libraries
import snowflake.snowpark as snowpark
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("📦 Libraries imported successfully")
print("="*60)

# 🔗 Snowflake Connection Setup
print("🔗 Establishing Snowflake connection...")

try:
    # Try our key pair authentication first (for local development in Cursor)
    from snowflake_connection import get_session
    session = get_session()
    print("✅ Connected using key pair authentication")
except ImportError:
    # Fallback for Snowflake environments (like Snowsight notebooks)
    try:
        from snowflake.snowpark.context import get_active_session
        session = get_active_session()
        print("✅ Using active Snowflake session")
    except:
        print("❌ No authentication method available")
        print("💡 Make sure you have snowflake_connection.py in your project")
        raise Exception("Unable to establish Snowflake connection")

# Verify connection and show current context
print(f"🏢 Account: {session.get_current_account()}")
print(f"👤 User: {session.get_current_user()}")
print(f"🏗️ Warehouse: {session.get_current_warehouse()}")
print(f"🗃️ Database: {session.get_current_database()}")
print(f"📁 Schema: {session.get_current_schema()}")

# Ensure we're in the right context for this demo
if session.get_current_database() != "STRAVA_DEMO_SAMPLE":
    session.use_database("STRAVA_DEMO_SAMPLE")
    print("🔄 Switched to STRAVA_DEMO_SAMPLE database")

if session.get_current_schema() != "RAW_DATA":
    session.use_schema("RAW_DATA")
    print("🔄 Switched to RAW_DATA schema")

print(f"🎯 Ready to work with {session.get_current_database()}.{session.get_current_schema()}")
print("="*60)


🔍 Checking and installing required packages...
✅ snowflake-snowpark-python already installed
✅ pandas already installed
✅ numpy already installed
✅ All packages ready!
📦 Libraries imported successfully
🔗 Establishing Snowflake connection...
✅ Snowflake session created successfully!
   Account: SFSENORTHAMERICA-ARYEUNG_AWS1
   User: aryeung
   Role: STRAVA_DEMO_ADMIN
   Warehouse: STRAVA_DEMO_WH
   Database: STRAVA_DEMO_SAMPLE
   Schema: RAW_DATA
✅ Connected using key pair authentication
🏢 Account: "SFSENORTHAMERICA-ARYEUNG_AWS1"
👤 User: "aryeung"
🏗️ Warehouse: "STRAVA_DEMO_WH"
🗃️ Database: "STRAVA_DEMO_SAMPLE"
📁 Schema: "RAW_DATA"
🔄 Switched to STRAVA_DEMO_SAMPLE database
🔄 Switched to RAW_DATA schema
🎯 Ready to work with "STRAVA_DEMO_SAMPLE"."RAW_DATA"


In [42]:
# Configuration
NUM_ATHLETES = 2000
NUM_ACTIVITIES = 10000
NUM_CHALLENGES = 10000
NUM_SUBSCRIPTIONS = 10000

# Date range - past 90 days with 50% in past 30 days
# Use current date to ensure data is always recent and relevant
end_date = datetime.now()
start_date_90d = end_date - timedelta(days=90)
start_date_30d = end_date - timedelta(days=30)

print(f"📅 Generating sample data from {start_date_90d.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"📅 50% of data will be in the past 30 days ({start_date_30d.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')})")
print(f"👥 Athletes: {NUM_ATHLETES:,}")
print(f"🏃 Activities: {NUM_ACTIVITIES:,}")
print(f"🏆 Challenges: {NUM_CHALLENGES:,}")
print(f"💳 Subscriptions: {NUM_SUBSCRIPTIONS:,}")


📅 Generating sample data from 2025-06-24 to 2025-09-22
📅 50% of data will be in the past 30 days (2025-08-23 to 2025-09-22)
👥 Athletes: 2,000
🏃 Activities: 10,000
🏆 Challenges: 10,000
💳 Subscriptions: 10,000


In [43]:
# Generate athlete profiles
print("👥 Generating athlete profiles...")

athlete_profiles = []
for athlete_id in range(1, NUM_ATHLETES + 1):
    # Base fitness level (affects all metrics)
    fitness_level = np.random.choice(['beginner', 'intermediate', 'advanced'], p=[0.4, 0.4, 0.2])
    
    # Activity frequency (activities per week)
    if fitness_level == 'beginner':
        activities_per_week = np.random.uniform(1, 3)
    elif fitness_level == 'intermediate':
        activities_per_week = np.random.uniform(2, 5)
    else:  # advanced
        activities_per_week = np.random.uniform(4, 8)
    
    # Base pace (km/h) - varies by fitness level
    if fitness_level == 'beginner':
        base_pace = np.random.uniform(8, 12)
    elif fitness_level == 'intermediate':
        base_pace = np.random.uniform(10, 16)
    else:  # advanced
        base_pace = np.random.uniform(14, 20)
    
    # Base heart rate (bpm) - inversely related to fitness
    if fitness_level == 'beginner':
        base_heartrate = np.random.uniform(140, 180)
    elif fitness_level == 'intermediate':
        base_heartrate = np.random.uniform(120, 160)
    else:  # advanced
        base_heartrate = np.random.uniform(100, 140)
    
    athlete_profiles.append({
        'athlete_id': athlete_id,
        'fitness_level': fitness_level,
        'activities_per_week': activities_per_week,
        'base_pace': base_pace,
        'base_heartrate': base_heartrate
    })

print(f"✅ Generated {len(athlete_profiles)} athlete profiles")


👥 Generating athlete profiles...
✅ Generated 2000 athlete profiles


In [44]:
# Generate activities data
print("🏃 Generating activities data...")

activities_data = []
activity_id = 1

for athlete_id in range(1, NUM_ATHLETES + 1):
    profile = athlete_profiles[athlete_id - 1]
    
    # Calculate number of activities for this athlete
    weeks_in_period = 13  # 90 days
    total_activities = int(profile['activities_per_week'] * weeks_in_period)
    
    # Add some randomness
    total_activities += np.random.randint(-2, 3)
    total_activities = max(1, total_activities)  # At least 1 activity
    
    for _ in range(total_activities):
        # Activity type distribution
        activity_type = np.random.choice(['Run', 'Ride', 'Walk', 'Swim'], p=[0.5, 0.3, 0.15, 0.05])
        
        # Date - 50% in past 30 days, 50% in past 90 days
        if np.random.random() < 0.5:
            # Past 30 days
            days_ago = np.random.randint(0, 30)
        else:
            # Past 90 days (but not in past 30)
            days_ago = np.random.randint(30, 90)
        
        start_date = end_date - timedelta(days=days_ago)
        
        # Distance (varies by activity type)
        if activity_type == 'Run':
            distance_km = np.random.uniform(2, 21)  # 2km to half marathon
        elif activity_type == 'Ride':
            distance_km = np.random.uniform(10, 100)  # 10km to 100km
        elif activity_type == 'Walk':
            distance_km = np.random.uniform(1, 10)  # 1km to 10km
        else:  # Swim
            distance_km = np.random.uniform(0.5, 5)  # 0.5km to 5km
        
        distance_meters = distance_km * 1000
        
        # Pace calculation (km/h)
        base_pace = profile['base_pace']
        
        # Adjust pace based on activity type
        if activity_type == 'Ride':
            pace_multiplier = np.random.uniform(1.5, 2.5)  # Bikes are faster
        elif activity_type == 'Walk':
            pace_multiplier = np.random.uniform(0.3, 0.6)  # Walks are slower
        elif activity_type == 'Swim':
            pace_multiplier = np.random.uniform(0.2, 0.4)  # Swimming is much slower
        else:  # Run
            pace_multiplier = np.random.uniform(0.8, 1.2)  # Slight variation
        
        pace_kmh = base_pace * pace_multiplier
        
        # Add some fraud indicators (5% of activities)
        is_fraud = np.random.random() < 0.05
        if is_fraud:
            if activity_type == 'Run':
                # Suspiciously fast running pace
                pace_kmh = np.random.uniform(20, 30)  # 20-30 km/h (elite runner pace)
            else:
                # Suspiciously high heart rate
                heartrate_multiplier = np.random.uniform(1.5, 2.0)
        
        # Calculate moving time based on pace and distance
        moving_time_hours = distance_km / pace_kmh
        moving_time_sec = int(moving_time_hours * 3600)
        
        # Elapsed time (slightly longer than moving time)
        elapsed_time_sec = moving_time_sec + np.random.randint(60, 300)  # 1-5 min extra
        
        # Heart rate
        base_heartrate = profile['base_heartrate']
        if is_fraud and activity_type != 'Run':
            average_heartrate = base_heartrate * heartrate_multiplier
        else:
            # Normal heart rate with some variation
            average_heartrate = base_heartrate + np.random.uniform(-10, 20)
        
        # Elevation gain (meters)
        if activity_type in ['Run', 'Walk']:
            elevation_gain = np.random.uniform(0, distance_km * 20)  # Up to 20m per km
        elif activity_type == 'Ride':
            elevation_gain = np.random.uniform(0, distance_km * 50)  # Up to 50m per km
        else:  # Swim
            elevation_gain = 0  # Swimming has no elevation
        
        # Generate a simple polyline (simplified)
        map_polyline = f"polyline_{athlete_id}_{activity_id}"
        
        activities_data.append({
            'activity_id': activity_id,
            'athlete_id': athlete_id,
            'activity_type': activity_type,
            'start_date_local': start_date,
            'distance_meters': round(distance_meters, 2),
            'moving_time_sec': moving_time_sec,
            'elapsed_time_sec': elapsed_time_sec,
            'total_elevation_gain_meters': round(elevation_gain, 1),
            'map_polyline': map_polyline,
            'average_heartrate': round(average_heartrate, 1)
        })
        
        activity_id += 1
        
        # Stop if we've reached the target number of activities
        if len(activities_data) >= NUM_ACTIVITIES:
            break
    
    if len(activities_data) >= NUM_ACTIVITIES:
        break

print(f"✅ Generated {len(activities_data)} activities")


🏃 Generating activities data...
✅ Generated 10000 activities


In [45]:
# Generate challenges data
print("🏆 Generating challenges data...")

challenges_data = []
challenge_id = 1

# Challenge partners
partners = ['Nike', 'Adidas', 'Under Armour', 'New Balance', 'Puma', 'Reebok', 'ASICS', 'Brooks']
challenge_types = ['Distance', 'Time', 'Frequency', 'Elevation', 'Speed']
statuses = ['active', 'completed', 'abandoned']

for _ in range(NUM_CHALLENGES):
    athlete_id = np.random.randint(1, NUM_ATHLETES + 1)
    partner = np.random.choice(partners)
    challenge_type = np.random.choice(challenge_types)
    
    # Join date (within the 90-day period)
    days_ago = np.random.randint(0, 90)
    join_date = end_date - timedelta(days=days_ago)
    
    # Status distribution
    status = np.random.choice(statuses, p=[0.3, 0.6, 0.1])
    
    # Progress (0-100%)
    if status == 'completed':
        progress = 100.0
    elif status == 'abandoned':
        progress = np.random.uniform(0, 50)
    else:  # active
        progress = np.random.uniform(0, 100)
    
    # Progress in meters (varies by challenge type)
    if challenge_type == 'Distance':
        target_meters = np.random.uniform(5000, 50000)  # 5km to 50km
    elif challenge_type == 'Time':
        target_meters = np.random.uniform(10000, 100000)  # 10km to 100km
    elif challenge_type == 'Frequency':
        target_meters = np.random.uniform(2000, 20000)  # 2km to 20km
    elif challenge_type == 'Elevation':
        target_meters = np.random.uniform(1000, 10000)  # 1km to 10km
    else:  # Speed
        target_meters = np.random.uniform(5000, 25000)  # 5km to 25km
    
    progress_meters = target_meters * (progress / 100)
    
    # Completion date (if completed)
    completion_date = None
    if status == 'completed':
        days_to_complete = np.random.randint(1, 30)
        completion_date = join_date + timedelta(days=days_to_complete)
    
    challenges_data.append({
        'challenge_id': challenge_id,
        'partner_name': partner,
        'athlete_id': athlete_id,
        'join_date': join_date,
        'status': status,
        'progress_meters': round(progress_meters, 2),
        'is_complete': status == 'completed',
        'completion_date': completion_date
    })
    
    challenge_id += 1

print(f"✅ Generated {len(challenges_data)} challenges")


🏆 Generating challenges data...
✅ Generated 10000 challenges


In [46]:
# Generate subscriptions data
print("💳 Generating subscriptions data...")

subscriptions_data = []

subscription_plans = ['Free', 'Premium', 'Pro']
features = ['Route Planning', 'Training Plans', 'Social Features', 'Analytics', 'Live Tracking', 'Challenges']

for user_id in range(1, NUM_SUBSCRIPTIONS + 1):
    # Subscription plan distribution
    plan = np.random.choice(subscription_plans, p=[0.6, 0.25, 0.15])
    
    # Plan start date (within the 90-day period)
    days_ago = np.random.randint(0, 90)
    plan_start_date = end_date - timedelta(days=days_ago)
    
    # Plan end date (if not free)
    plan_end_date = None
    if plan != 'Free':
        if plan == 'Premium':
            plan_end_date = plan_start_date + timedelta(days=30)  # Monthly
        else:  # Pro
            plan_end_date = plan_start_date + timedelta(days=365)  # Yearly
    
    # Trial status (only for paid plans)
    is_trial = False
    if plan != 'Free' and np.random.random() < 0.3:  # 30% chance of trial
        is_trial = True
    
    # Last feature used
    last_feature_used = np.random.choice(features)
    
    # Login activity (last 30 days)
    total_logins_l30d = np.random.poisson(15)  # Average 15 logins per month
    
    # Churn risk score (0-100)
    if plan == 'Free':
        churn_risk = np.random.uniform(60, 90)  # High churn risk for free users
    elif is_trial:
        churn_risk = np.random.uniform(40, 70)  # Medium-high for trial users
    else:
        churn_risk = np.random.uniform(10, 40)  # Lower risk for paying customers
    
    subscriptions_data.append({
        'user_id': user_id,  # Note: using user_id as user_id for consistency
        'subscription_plan': plan,
        'plan_start_date': plan_start_date,
        'plan_end_date': plan_end_date,
        'is_trial': is_trial,
        'last_feature_used': last_feature_used,
        'total_logins_l30d': total_logins_l30d,
        'churn_risk_score': round(churn_risk, 2)
    })

print(f"✅ Generated {len(subscriptions_data)} subscriptions")


💳 Generating subscriptions data...
✅ Generated 10000 subscriptions


## 📊 Data Loading Section

Now we'll load the generated data into Snowflake tables. This section:
- Creates DataFrames from the generated data
- Saves data to the Snowflake stage
- Loads data into the ACTIVITIES, CHALLENGES, and SUBSCRIPTIONS tables
- Verifies the data was loaded correctly


In [47]:
# Create DataFrames
print("📊 Creating DataFrames...")

activities_df = pd.DataFrame(activities_data)
print(f"✅ Created activities DataFrame: {len(activities_df)} rows")

challenges_df = pd.DataFrame(challenges_data)
print(f"✅ Created challenges DataFrame: {len(challenges_df)} rows")

subscriptions_df = pd.DataFrame(subscriptions_data)
print(f"✅ Created subscriptions DataFrame: {len(subscriptions_df)} rows")


📊 Creating DataFrames...
✅ Created activities DataFrame: 10000 rows
✅ Created challenges DataFrame: 10000 rows
✅ Created subscriptions DataFrame: 10000 rows


In [48]:
# Track successful table loading
tables_loaded = []

# Load activities data
print("🔄 Loading activities data into table...")
try:
    # Clear existing data and insert new data
    session.sql("TRUNCATE TABLE ACTIVITIES").collect()
    
    # Insert data using SQL
    if activities_data:
        # Convert data to SQL INSERT statements
        insert_values = []
        for row in activities_data:
            values = f"({row['activity_id']}, {row['athlete_id']}, '{row['activity_type']}', '{row['start_date_local']}', {row['distance_meters']}, {row['moving_time_sec']}, {row['elapsed_time_sec']}, {row['total_elevation_gain_meters']}, '{row['map_polyline']}', {row['average_heartrate']})"
            insert_values.append(values)
        
        # Insert in batches to avoid SQL statement length limits
        batch_size = 1000
        for i in range(0, len(insert_values), batch_size):
            batch = insert_values[i:i+batch_size]
            insert_sql = f"INSERT INTO ACTIVITIES VALUES {','.join(batch)}"
            session.sql(insert_sql).collect()
    
    print(f"✅ Activities data loaded successfully! ({len(activities_data)} rows)")
    tables_loaded.append("ACTIVITIES")
    
except Exception as e:
    print(f"❌ Error loading activities: {e}")


🔄 Loading activities data into table...
✅ Activities data loaded successfully! (10000 rows)


In [49]:
# Load challenges data
print("🔄 Loading challenges data into table...")
try:
    # Clear existing data and insert new data
    session.sql("TRUNCATE TABLE CHALLENGES").collect()
    
    # Insert data using SQL
    if challenges_data:
        # Convert data to SQL INSERT statements
        insert_values = []
        for row in challenges_data:
            completion_date = f"'{row['completion_date']}'" if row['completion_date'] else "NULL"
            values = f"({row['challenge_id']}, '{row['partner_name']}', {row['athlete_id']}, '{row['join_date']}', '{row['status']}', {row['progress_meters']}, {str(row['is_complete']).upper()}, {completion_date})"
            insert_values.append(values)
        
        # Insert in batches to avoid SQL statement length limits
        batch_size = 1000
        for i in range(0, len(insert_values), batch_size):
            batch = insert_values[i:i+batch_size]
            insert_sql = f"INSERT INTO CHALLENGES VALUES {','.join(batch)}"
            session.sql(insert_sql).collect()
    
    print(f"✅ Challenges data loaded successfully! ({len(challenges_data)} rows)")
    tables_loaded.append("CHALLENGES")
    
except Exception as e:
    print(f"❌ Error loading challenges: {e}")


🔄 Loading challenges data into table...
✅ Challenges data loaded successfully! (10000 rows)


In [50]:
# Load subscriptions data
print("🔄 Loading subscriptions data into table...")
try:
    # Clear existing data and insert new data
    session.sql("TRUNCATE TABLE SUBSCRIPTIONS").collect()
    
    # Insert data using SQL
    if subscriptions_data:
        # Convert data to SQL INSERT statements
        insert_values = []
        for row in subscriptions_data:
            plan_end_date = f"'{row['plan_end_date']}'" if row['plan_end_date'] else "NULL"
            last_feature = f"'{row['last_feature_used']}'" if row['last_feature_used'] else "NULL"
            values = f"({row['user_id']}, '{row['subscription_plan']}', '{row['plan_start_date']}', {plan_end_date}, {str(row['is_trial']).upper()}, {last_feature}, {row['total_logins_l30d']}, {row['churn_risk_score']})"
            insert_values.append(values)
        
        # Insert in batches to avoid SQL statement length limits
        batch_size = 1000
        for i in range(0, len(insert_values), batch_size):
            batch = insert_values[i:i+batch_size]
            insert_sql = f"INSERT INTO SUBSCRIPTIONS VALUES {','.join(batch)}"
            session.sql(insert_sql).collect()
    
    print(f"✅ Subscriptions data loaded successfully! ({len(subscriptions_data)} rows)")
    tables_loaded.append("SUBSCRIPTIONS")
    
except Exception as e:
    print(f"❌ Error loading subscriptions: {e}")


🔄 Loading subscriptions data into table...
✅ Subscriptions data loaded successfully! (10000 rows)


In [51]:
# Verify data loaded (only for successfully loaded tables)
if tables_loaded:
    print(f"\n📊 Data verification for {len(tables_loaded)} successfully loaded tables:")
    
    # Build dynamic query for row counts
    union_queries = []
    for table in tables_loaded:
        union_queries.append(f"SELECT '{table}' as table_name, COUNT(*) as row_count FROM {table}")
    
    if union_queries:
        row_counts_query = " UNION ALL ".join(union_queries)
        try:
            row_counts = session.sql(row_counts_query).collect()
            for row in row_counts:
                print(f"  - {row['TABLE_NAME']}: {row['ROW_COUNT']} rows")
        except Exception as e:
            print(f"❌ Error verifying row counts: {e}")
    
    # Show sample data for each table
    print("\n📋 Sample data preview:")
    for table in tables_loaded:
        try:
            print(f"\n{table} sample:")
            session.table(table).limit(3).show()
        except Exception as e:
            print(f"❌ Error showing sample data for {table}: {e}")
    
    print(f"\n✅ Successfully loaded {len(tables_loaded)} tables!")
    print("You can now run your ML demo notebook!")
else:
    print("\n❌ No tables were successfully loaded. Please check the error messages above.")



📊 Data verification for 3 successfully loaded tables:
  - ACTIVITIES: 10000 rows
  - CHALLENGES: 10000 rows
  - SUBSCRIPTIONS: 10000 rows

📋 Sample data preview:

ACTIVITIES sample:
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"ACTIVITY_ID"  |"ATHLETE_ID"  |"ACTIVITY_TYPE"  |"START_DATE_LOCAL"          |"DISTANCE_METERS"  |"MOVING_TIME_SEC"  |"ELAPSED_TIME_SEC"  |"TOTAL_ELEVATION_GAIN_METERS"  |"MAP_POLYLINE"  |"AVERAGE_HEARTRATE"  |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|1              |1             |Ride             |2025-08-28 12:39:39.088236  |19175.73           |3316               |3395                |664.9                          |polyline_

In [52]:
# Summary statistics
print("📊 Complete Data Generation and Loading Summary:")
print(f"👥 Athletes: {NUM_ATHLETES:,}")
print(f"🏃 Activities: {len(activities_data):,}")
print(f"🏆 Challenges: {len(challenges_data):,}")
print(f"💳 Subscriptions: {len(subscriptions_data):,}")
print(f"📅 Date range: {start_date_90d.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"📊 Tables loaded: {len(tables_loaded)}")
print(f"\n✅ All data generated and loaded successfully!")
print(f"\n📝 Next steps:")
print(f"  1. Run your ML demo notebook (STRAVA_ML_DEMO_2.ipynb)")
print(f"  2. The data is now ready for machine learning analysis")
print(f"  3. You can start building fraud detection models!")


📊 Complete Data Generation and Loading Summary:
👥 Athletes: 2,000
🏃 Activities: 10,000
🏆 Challenges: 10,000
💳 Subscriptions: 10,000
📅 Date range: 2025-06-24 to 2025-09-22
📊 Tables loaded: 3

✅ All data generated and loaded successfully!

📝 Next steps:
  1. Run your ML demo notebook (STRAVA_ML_DEMO_2.ipynb)
  2. The data is now ready for machine learning analysis
  3. You can start building fraud detection models!
